In [80]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import certifi
import os
from dotenv import load_dotenv
from urllib.parse import quote_plus

load_dotenv()

True

In [81]:
username = os.getenv("MONGODB_USERNAME")
password = os.getenv("MONGODB_PASSWORD")
cluster = os.getenv("MONGODB_CLUSTER")

password = quote_plus(password)

mongo_uri = (
    f"mongodb+srv://{username}:{password}@{cluster}/"
    f"?appName=Cluster0"
)

client = MongoClient(
    mongo_uri,
    server_api=ServerApi("1"),
    tlsCAFile=certifi.where()
)

client.admin.command("ping")

print("Conexão com MongoDB realizada com sucesso!")

Conexão com MongoDB realizada com sucesso!


In [82]:
db = client["IBGE"]

print(db.name)

IBGE


In [83]:
data = db['PNADC'].find()
print(db)

Database(MongoClient(host=['ac-8v03p3t-shard-00-01.cvgxdpf.mongodb.net:27017', 'ac-8v03p3t-shard-00-00.cvgxdpf.mongodb.net:27017', 'ac-8v03p3t-shard-00-02.cvgxdpf.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='Cluster0', authsource='admin', replicaset='atlas-msg7du-shard-0', tls=True, server_api=<pymongo.server_api.ServerApi object at 0x7f8078cf5340>, tlscafile='/Users/tsilva/Documents/GRADUACAO - BANCO DE DADOS/2-PERIODO/Engenharia_Dados/ProjEngDados202602-BD/.venv/lib/python3.9/site-packages/certifi/cacert.pem'), 'IBGE')


In [84]:
serie = data[0]['resultados'][0]['series'][0]['serie']
print(serie)

{'201201': '9.6', '201202': '8.3', '201203': '9.4', '201204': '9.2', '201301': '10.7', '201302': '9.7', '201303': '8.5', '201304': '7.4', '201401': '8.8', '201402': '8.0', '201403': '8.5', '201404': '7.7', '201501': '8.2', '201502': '9.2', '201503': '11.3', '201504': '11.1', '201601': '13.4', '201602': '14.2', '201603': '15.5', '201604': '15.9', '201701': '17.3', '201702': '19.0', '201703': '18.1', '201704': '17.0', '201801': '17.9', '201802': '17.1', '201803': '17.0', '201804': '15.6', '201901': '16.3', '201902': '16.1', '201903': '16.0', '201904': '14.2', '202001': '14.8', '202002': '...', '202003': '...', '202004': '...', '202101': '...', '202102': '...', '202103': '...', '202104': '...', '202201': '...', '202202': '13.6', '202203': '14.0', '202204': '12.3', '202301': '14.1', '202302': '14.2', '202303': '13.3', '202304': '12.0', '202401': '12.4', '202402': '11.6', '202403': '10.6', '202404': '10.3', '202501': '11.6', '202502': '10.4', '202503': '10.0', '202504': '8.8', '202601': '9.

In [85]:
import pandas as pd

df = pd.DataFrame.from_dict(serie, orient='index', columns=['valor'])
df.index.name = 'periodo'
df = df.reset_index()
df

,periodo,valor
0,201201,9.6
1,201202,8.3
2,201203,9.4
3,201204,9.2
4,201301,10.7
5,201302,9.7
6,201303,8.5
7,201304,7.4
8,201401,8.8
9,201402,8.0


In [86]:
#df.head()
#df.tail()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   periodo  57 non-null     object
 1   valor    57 non-null     object
dtypes: object(2)
memory usage: 1.0+ KB


In [87]:
df['valor'] = df['valor'].replace('...', '0')
df['valor'] = df['valor'].astype(float)


In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   periodo  57 non-null     object 
 1   valor    57 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.0+ KB


In [91]:
df['ano'] = df['periodo'].str[:4]
df['tri'] = df['periodo'].str[-1:]
df['periodo'] = pd.PeriodIndex(df['ano']+'Q'+df['tri'], freq='Q')

AttributeError: Can only use .str accessor with string values!

In [ ]:
df['periodo'] = df['periodo'].dt.to_timestamp()
df

ValueError: Columns must be same length as key

In [1]:
import sqlite3

conn = sqlite3.connect('IBGE.db')

df.to_sql('pnadc-total', conn, if_exists='replace', index=False)

NameError: name 'df' is not defined